# Fine-tune the cross-encoder reranker (Colab)

Runtime > Change runtime type > **T4 GPU**. Run the cells **in order**, one at a time. Only the **dev** split is evaluated; test is never touched.

You need three small files from your Mac (about 2.7 MB in total, in `~/Documents/colab-upload/`):
`ce_bm25.json`, `ce_dense_D.json` and `dev_hybrid_D.json`. The bi-encoder checkpoint is **not** needed.

In [ ]:
%%bash
set -e
[ -d /content/financial-qa-retrieval ] || git clone https://github.com/IbrahimKhan25/financial-qa-retrieval.git
cd /content/financial-qa-retrieval
git pull
python --version

In [ ]:
%cd /content/financial-qa-retrieval
!pip install -q -e . bm25s scipy mlflow
import torch
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))

In [ ]:
!python scripts/download_data.py

## Upload the three files
Click "Choose files" and select all three at once. The cell checks that they landed in the right place.

In [ ]:
import os, shutil
from google.colab import files
os.makedirs("data/negatives", exist_ok=True)
os.makedirs("data/candidates", exist_ok=True)
dest = {"ce_bm25.json": "data/negatives", "ce_dense_D.json": "data/negatives", "dev_hybrid_D.json": "data/candidates"}
for name in files.upload():
    if name in dest:
        shutil.move(name, f"{dest[name]}/{name}")
missing = [n for n, d in dest.items() if not os.path.exists(f"{d}/{n}")]
assert not missing, f"missing or misnamed: {missing} (rename e.g. 'ce_bm25 (1).json' back to 'ce_bm25.json' and re-run)"
print("all three files in place")

## Train
Epoch 0 in the log is the **off-the-shelf** reranker on the same dev candidates. Sanity check: its NDCG@10 should be about **0.365**
(my local run gave 0.3652). If it is far from that, stop and tell me. Then it fine-tunes for 2 epochs; expect roughly 15 to 25 minutes.

In [ ]:
!python scripts/train_cross_encoder.py --config configs/cross_encoder_ft_mixed.yaml

## Download the results and the checkpoint
Two downloads: a small results file and the fine-tuned checkpoint (about 90 MB). Do not close the tab before both finish.

In [ ]:
!zip -j results_reranker_mixed.zip results/train_ce_*.json
!zip -r -q checkpoint_ce_mixed.zip checkpoints/cross_encoder_ft_mixed
from google.colab import files
files.download("results_reranker_mixed.zip")
files.download("checkpoint_ce_mixed.zip")